In [38]:
import os
from databricks.connect import DatabricksSession

# Initialize Spark locally when needed
if "spark" not in globals():
    spark = DatabricksSession.builder.serverless().getOrCreate()

#NotebookParameters
if "dbutils" in globals():
    #Execute in Databricks
    dbutils.widgets.dropdown(
        "environment", 
        "dev", 
        ["dev", "staging", "prod"]
    )

    dbutils.widgets.text("organization", "databricks")

    environment = dbutils.widgets.get("environment")
    organization = dbutils.widgets.get("organization").strip()
else:
    #Execute in VSCode with Databricks Connect
    environment = os.getenv("LAKEHOUSE_ENV", "dev")
    organization = os.getenv("GITHUB_ORGANIZATION", "databricks").strip()

catalogs = {
    "dev": "opensource_dev",
    "staging": "opensource_staging",
    "prod": "opensource_prod"
}

catalog = catalogs[environment]
schema_name = "github_analytics"

# Define Silver source and Gold target tables
silver_table = f"{catalog}.{schema_name}.silver_github_repositories"
gold_table = f"{catalog}.{schema_name}.gold_github_repositories"

print(f"Source Table: {silver_table}")
print(f"Target Table: {gold_table}")

Source Table: opensource_dev.github_analytics.silver_github_repositories
Target Table: opensource_dev.github_analytics.gold_github_repositories


In [39]:
from pyspark.sql import functions as F

# Read new and updated repositories from Silver
silver_df = (
        spark.read.table(silver_table)
        .filter(F.col("organization") == organization)
)

print(f"Silver records to process: {silver_df.count()}")

silver_df.groupBy("change_type").count().show()

silver_df.select(
    "id",
    "full_name",
    "description",
    "language",
    "archived",
    "change_type",
    "ingestion_ts"
).show(10, truncate=False)

Silver records to process: 253
+-----------+-----+
|change_type|count|
+-----------+-----+
|        NEW|  253|
+-----------+-----+

+----------+---------------------------------+---------------------------------------------------------------------------------------------+----------+--------+-----------+--------------------------+
|id        |full_name                        |description                                                                                  |language  |archived|change_type|ingestion_ts              |
+----------+---------------------------------+---------------------------------------------------------------------------------------------+----------+--------+-----------+--------------------------+
|757044536 |databricks/alertmanager          |Prometheus Alertmanager                                                                      |Go        |false   |NEW        |2026-11-14 16:54:29.583757|
|710920829 |databricks/api-linter            |A linter for APIs defi

In [40]:
#Initiliaze Gold load
if not spark.catalog.tableExists(gold_table):

    gold_df = (
        silver_df
        .withColumn(
            "repository_sk",
            F.sha2(
                F.concat_ws(
                    "_",
                    F.col("id"),
                    F.col("snapshot_id")
                ),
                256
            )
        )
        .withColumn("valid_from", F.col("ingestion_ts"))
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
        .withColumn("current", F.lit(True))
        .select(
            "repository_sk",
            "id",
            "name",
            "full_name",
            "description",
            "language",
            "archived",
            "default_branch",
            "organization",
            "snapshot_id",
            "valid_from",
            "valid_to",
            "current"
        )
    )

    # Create the initial Gold Delta table
    (
        gold_df.write
        .format("delta")
        .saveAsTable(gold_table)
    )

    print(f"Gold table created: {gold_table}")

else:
    print(f"Gold table already exists: {gold_table}")

Gold table already exists: opensource_dev.github_analytics.gold_github_repositories


In [41]:
#Verify the initial load
df_gold = spark.read.table(gold_table)

print(f"Total Gold records: {df_gold.count()}")

df_gold.groupBy("current").count().show()

df_gold.select(
    "repository_sk",
    "id",
    "full_name",
    "valid_from",
    "valid_to",
    "current"
).show(10, truncate=False)


Total Gold records: 253
+-------+-----+
|current|count|
+-------+-----+
|   true|  253|
+-------+-----+

+----------------------------------------------------------------+----------+---------------------------------+--------------------------+--------+-------+
|repository_sk                                                   |id        |full_name                        |valid_from                |valid_to|current|
+----------------------------------------------------------------+----------+---------------------------------+--------------------------+--------+-------+
|1e529fbe43b7c54dbd03deeb27bf10812b7bbea226e114121a34c46df0ea4e59|757044536 |databricks/alertmanager          |2026-11-14 16:54:29.583757|NULL    |true   |
|c3f75857bad8806a8a1b55478ee7630b2cc42c2927be261113762266fbe549e7|710920829 |databricks/api-linter            |2026-11-14 16:54:29.583757|NULL    |true   |
|68da9fa6c27f513784f8a87a4d1a536bc977d8abcc96f2b79a8c18b8cc79c736|811440846 |databricks/app-templates         |2026

In [42]:
from delta.tables import DeltaTable

#Retrieve the latest processed Silver snapshot
latest_snapshot = (
    silver_df
    .select("snapshot_id", "ingestion_ts")
    .distinct()
    .orderBy(F.col("ingestion_ts").desc())
    .first()
)

if latest_snapshot is None:
    raise ValueError("No Silver snapshots available")

silver_latest = silver_df.filter(
  F.col("snapshot_id") == latest_snapshot["snapshot_id"]
)

#Load Gold Delta table
delta_table_gold = DeltaTable.forName(spark, gold_table)

# Attributes monitored for SCD Type 2
tracked_columns = [
    "name",
    "full_name",
    "description",
    "language",
    "archived",
    "default_branch"
]

# Build null-safe change detection condition
change_condition = F.lit(False)

for column in tracked_columns:
    change_condition = change_condition | (
      ~F.col(f"target.{column}").eqNullSafe(
         F.col(f"source.{column}")
      )
    )


# Retrieve current Gold records
gold_current = (
    delta_table_gold.toDF()
    .filter(F.col("current") == True)
)

# Identify existing repositories with changed attributes
df_updated = (
    silver_latest.alias("source")
    .join(
        gold_current.alias("target"),
        (F.col("source.id") == F.col("target.id")) &
        (F.col("source.organization") == F.col("target.organization")),
        "inner"
    )
    .filter(
        change_condition &
        (F.col("source.ingestion_ts") > F.col("target.valid_from"))
    )
    .select("source.*")
)

print(f"Updated repositories: {df_updated.count()}")



Updated repositories: 0


In [43]:
#Stage records for SCD2 Type2 MERGE
staged_df = (
    silver_latest.withColumn(
        "merge_id",
        F.col("id")
        )
    .unionByName(
        df_updated.withColumn(
            "merge_id",
            F.lit(None).cast("string")
        )   
    )
)

print(f"Records staged for MERGE: {staged_df.count()}")

Records staged for MERGE: 253


In [ ]:
#Apply SCD Type 2 changes to the Gold Delta Table
(
    delta_table_gold.alias("target")
    .merge(
        staged_df.alias("source"), "" 
        """
        target.id = source.merge_id
        AND target.organization = source.organization
        AND target.current = True
        """
    )
    .whenMatchedUpdate(
        condition = (
            change_condition &
           (F.col("source.ingestion_ts") > F.col("target.valid_from"))
        ),
        set = {
            "valid_to": F.col("source.ingestion_ts"),
            "current": F.lit(False)
        }
    )
    .whenNotMatchedInsert(
        values = {
            "repository_sk": F.sha2(
                F.concat_ws(
                    "_",
                    F.col("source.id"),
                    F.col("source.snapshot_id")
                ),
                256
            ),
            "id": F.col("source.id"),
            "name": F.col("source.name"),
            "full_name": F.col("source.full_name"),
            "description": F.col("source.description"),
            "language": F.col("source.language"),
            "archived": F.col("source.archived"),
            "default_branch": F.col("source.default_branch"),
            "organization": F.col("source.organization"),
            "snapshot_id": F.col("source.snapshot_id"),
            "valid_from": F.col("source.ingestion_ts"),
            "valid_to": F.lit(None).cast("timestamp"),
            "current": F.lit(True)
        }
    )
    .execute()
)

print("Gold SCD Type 2 MERGE completed")

Gold SCD Type 2 MERGE completed


In [45]:
#Verify Gold SCD Type 2 results
df_gold = spark.read.table(gold_table)

print(f"Total Gold records: {df_gold.count()}")

df_gold.groupBy("current").count().show()

#Check for multiple current versions of the same repository
duplicate_current = (
    df_gold
    .filter(F.col("current") == True)
    .groupBy("id", "organization")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Repositories with multiple current versions: {duplicate_current.count()}")

Total Gold records: 253
+-------+-----+
|current|count|
+-------+-----+
|   true|  253|
+-------+-----+

Repositories with multiple current versions: 0


In [46]:

# Read historical repository metrics from Bronze
bronze_table = f"{catalog}.{schema_name}.bronze_github_repositories"

df_metrics = (
    spark.read.table(bronze_table)
    .filter(F.col("organization") == organization)
    .select(
        "id",
        "full_name",
        "organization",
        "stargazers_count",
        "forks_count",
        "snapshot_id",
        "ingestion_ts"
    )
)

# Verify metric evolution for a specific repository
(
    df_metrics
    .filter(F.col("full_name") == "databricks/app-templates")
    .orderBy("ingestion_ts")
    .show(truncate=False)
)


+---------+------------------------+------------+----------------+-----------+------------------------------------+--------------------------+
|id       |full_name               |organization|stargazers_count|forks_count|snapshot_id                         |ingestion_ts              |
+---------+------------------------+------------+----------------+-----------+------------------------------------+--------------------------+
|811440846|databricks/app-templates|databricks  |221             |166        |943612ea-bc21-4671-b03e-b437e6f2482b|2026-11-14 16:54:29.583757|
|811440846|databricks/app-templates|databricks  |222             |166        |6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|
+---------+------------------------+------------+----------------+-----------+------------------------------------+--------------------------+



In [47]:

# Define the Gold metrics table
metrics_table = f"{catalog}.{schema_name}.gold_github_repository_metrics"

# Prepare historical repository metrics
df_metrics_gold = (
    df_metrics
    .withColumn("snapshot_date", F.to_date("ingestion_ts"))
    .select(
        "id",
        "full_name",
        "organization",
        "stargazers_count",
        "forks_count",
        "snapshot_id",
        "snapshot_date",
        "ingestion_ts"
    )
)

# Identify records not yet loaded
if spark.catalog.tableExists(metrics_table):

    existing_metrics = (
        spark.read.table(metrics_table)
        .select("organization", "id", "snapshot_id")
        .distinct()
    )

    df_to_insert = df_metrics_gold.join(
        existing_metrics,
        on=["organization", "id", "snapshot_id"],
        how="left_anti"
    )

else:
    df_to_insert = df_metrics_gold

# Append only new metric snapshots
records_to_insert = df_to_insert.count()

if records_to_insert > 0:
    (
        df_to_insert.write
        .format("delta")
        .mode("append")
        .saveAsTable(metrics_table)
    )

print(f"Gold metrics table: {metrics_table}")
print(f"Records inserted: {records_to_insert}")


Gold metrics table: opensource_dev.github_analytics.gold_github_repository_metrics
Records inserted: 0


In [48]:
df_check = spark.read.table(metrics_table)

print(f"Total metric records: {df_check.count()}")

df_check.groupBy("snapshot_date").count() \
    .orderBy("snapshot_date") \
    .show()

Total metric records: 506
+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2026-11-14|  253|
|   2026-11-15|  253|
+-------------+-----+



In [49]:
from pyspark.sql.window import Window

#Read Gold repository metrics
df_metrics = spark.read.table(metrics_table)

#Define a window for each repository ordered by ingestion time
w = (
    Window
    .partitionBy("organization", "id")
    .orderBy("ingestion_ts", "snapshot_id")
)

#Calculate metric variations between consecutive snapshots
df_metrics_trends = (
    df_metrics
    .withColumn(
        "previous_stars",
        F.lag("stargazers_count").over(w)
    )
    .withColumn(
        "previous_forks",
        F.lag("forks_count").over(w)
    )
    .withColumn(
        "stars_delta",
        F.col("stargazers_count") - F.col("previous_stars")
    )
    .withColumn(
        "forks_delta",
        F.col("forks_count") - F.col("previous_forks")
    )
)

#Display metric evolution for a sample repository
(
    df_metrics_trends
    .filter(F.col("full_name") ==  "databricks/app-templates")
    .select(
        "full_name",
        "snapshot_date",
        "stargazers_count",
        "previous_stars",
        "stars_delta",
        "forks_count",
        "previous_forks",
        "forks_delta"
    )
    .orderBy("snapshot_date")
    .show(truncate=False)
)

+------------------------+-------------+----------------+--------------+-----------+-----------+--------------+-----------+
|full_name               |snapshot_date|stargazers_count|previous_stars|stars_delta|forks_count|previous_forks|forks_delta|
+------------------------+-------------+----------------+--------------+-----------+-----------+--------------+-----------+
|databricks/app-templates|2026-11-14   |221             |NULL          |NULL       |166        |NULL          |NULL       |
|databricks/app-templates|2026-11-15   |222             |221           |1          |166        |166           |0          |
+------------------------+-------------+----------------+--------------+-----------+-----------+--------------+-----------+



In [50]:
# Identify repositories with the highest star growth
(
    df_metrics_trends
    .filter(F.col("stars_delta") > 0)
    .select(
        "full_name",
        "snapshot_date",
        "stars_delta",
        "forks_delta"
    )
    .orderBy(F.col("stars_delta").desc())
    .show(10, truncate=False)
)


+----------------------------------+-------------+-----------+-----------+
|full_name                         |snapshot_date|stars_delta|forks_delta|
+----------------------------------+-------------+-----------+-----------+
|databricks/tmm                    |2026-11-15   |1          |0          |
|databricks/databricks-agent-skills|2026-11-15   |1          |1          |
|databricks/dicer                  |2026-11-15   |1          |0          |
|databricks/cli                    |2026-11-15   |1          |1          |
|databricks/app-templates          |2026-11-15   |1          |0          |
+----------------------------------+-------------+-----------+-----------+



In [51]:
# Read Gold table after SCD Type 2 MERGE
df_gold = spark.read.table(gold_table)

# Check mandatory fields
invalid_records = df_gold.filter(
    F.col("repository_sk").isNull() |
    F.col("id").isNull() |
    F.col("organization").isNull() |
    F.col("valid_from").isNull() |
    F.col("current").isNull()
)

# Check duplicate surrogate keys
duplicate_keys = (
    df_gold
    .groupBy("repository_sk")
    .count()
    .filter(F.col("count") > 1)
)

# Check multiple current versions per repository
duplicate_current = (
    df_gold
    .filter(F.col("current") == True)
    .groupBy("id", "organization")
    .count()
    .filter(F.col("count") > 1)
)

# Check invalid SCD Type 2 validity periods
invalid_validity = df_gold.filter(
    (
        (F.col("current") == True) &
        F.col("valid_to").isNotNull()
    ) |
    (
        (F.col("current") == False) &
        (
            F.col("valid_to").isNull() |
            (F.col("valid_to") <= F.col("valid_from"))
        )
    )
)

# Execute Data Quality checks
invalid_count = invalid_records.count()
duplicate_keys_count = duplicate_keys.count()
duplicate_current_count = duplicate_current.count()
invalid_validity_count = invalid_validity.count()

print(f"Invalid records: {invalid_count}")
print(f"Duplicate surrogate keys: {duplicate_keys_count}")
print(f"Multiple current versions: {duplicate_current_count}")
print(f"Invalid validity periods: {invalid_validity_count}")

# Stop execution if Data Quality checks fail
if (
    invalid_count > 0 or
    duplicate_keys_count > 0 or
    duplicate_current_count > 0 or
    invalid_validity_count > 0
):
    raise ValueError("Gold SCD Type 2 data quality validation failed")

print("Gold SCD Type 2 data quality validation passed")

Invalid records: 0
Duplicate surrogate keys: 0
Multiple current versions: 0
Invalid validity periods: 0
Gold SCD Type 2 data quality validation passed


In [52]:
# Read Gold repository metrics
df_metrics_check = spark.read.table(metrics_table)

# Check mandatory fields
invalid_records = df_metrics_check.filter(
    F.col("id").isNull() |
    F.col("organization").isNull() |
    F.col("snapshot_id").isNull() |
    F.col("ingestion_ts").isNull() |
    F.col("snapshot_date").isNull() |
    F.col("stargazers_count").isNull() |
    F.col("forks_count").isNull()
)

# Check duplicate repository metrics within the same snapshot
duplicate_metrics = (
    df_metrics_check
    .groupBy("organization", "id", "snapshot_id")
    .count()
    .filter(F.col("count") > 1)
)

# Check negative metric values
invalid_metrics = df_metrics_check.filter(
    (F.col("stargazers_count") < 0) |
    (F.col("forks_count") < 0)
)

# Check snapshot date consistency
invalid_dates = df_metrics_check.filter(
    F.col("snapshot_date") != F.to_date("ingestion_ts")
)

# Execute Data Quality checks
total_records = df_metrics_check.count()
invalid_count = invalid_records.count()
duplicate_count = duplicate_metrics.count()
invalid_metrics_count = invalid_metrics.count()
invalid_dates_count = invalid_dates.count()

print(f"Total metric records: {total_records}")
print(f"Invalid records: {invalid_count}")
print(f"Duplicate metric keys: {duplicate_count}")
print(f"Negative metric values: {invalid_metrics_count}")
print(f"Invalid snapshot dates: {invalid_dates_count}")

# Stop execution if Data Quality checks fail
if (
    total_records == 0 or
    invalid_count > 0 or
    duplicate_count > 0 or
    invalid_metrics_count > 0 or
    invalid_dates_count > 0
):
    raise ValueError("Gold metrics data quality validation failed")

print("Gold metrics data quality validation passed")


Total metric records: 506
Invalid records: 0
Duplicate metric keys: 0
Negative metric values: 0
Invalid snapshot dates: 0
Gold metrics data quality validation passed
